# Kronwald Dissipative Squeezing

This notebook uses `kronwald_optimizer.py` to demonstrate the Kronwald scheme from:

> Kronwald & Marquardt, *Arbitrarily Large Steady-State Bosonic Squeezing via Dissipation*, PRL 111, 133601 (2013).

**Setup:** An optomechanical cavity driven by two tones creates both a beamsplitter coupling $g$ (red sideband) and a squeezing coupling $\nu$ (blue sideband). In steady state the mechanical mode is squeezed below vacuum:

$$\sigma_{xx} = \frac{g-\nu}{2(g+\nu)} < \frac{1}{2} \quad\text{(squeezed)}$$

The squeezing parameter is $r = \text{atanh}(\nu/g)$ — arbitrarily large as $\nu \to g$.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from kronwald_optimizer import (
    solve_lyapunov, mechanical_cov,
    squeezed_vacuum_cov, squeezing_from_cov,
    kronwald_squeezing, kronwald_params_from_r,
    optimize,
)

plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

## 1. Verify the vacuum state (sanity check)

With $g = \nu = 0$ (no coupling), the mechanical mode should be in vacuum: $\sigma_{xx} = \sigma_{pp} = 1/2$.

In [ ]:
sigma_vac = np.array(solve_lyapunov(g=0.0, nu=0.0, kappa=1.0, gamma=0.01, n_th=0.0))
sm_vac = np.array(mechanical_cov(jnp.array(sigma_vac)))
print('Mechanical covariance (vacuum):')
print(sm_vac)
print(f'\n<x²> = {sm_vac[0,0]:.4f}  (expected 0.5000)')
print(f'<p²> = {sm_vac[1,1]:.4f}  (expected 0.5000)')

## 2. Squeezing vs coupling ratio $\nu/g$

Fix $g = 1$ and sweep $\nu/g$ from 0 to ~1. Compare numerical result to analytical Kronwald formula.

In [ ]:
gamma = 0.001    # small mechanical decay
kappa = 1.0
g     = 1.0

ratio_vals = np.linspace(0.0, 0.98, 80)
sigma_xx_vals = []
sigma_pp_vals = []

for ratio in ratio_vals:
    nu = g * ratio
    sm = np.array(mechanical_cov(jnp.array(
        solve_lyapunov(g, nu, kappa=kappa, gamma=gamma, n_th=0.0))))
    sigma_xx_vals.append(sm[0, 0])
    sigma_pp_vals.append(sm[1, 1])

# Analytical Kronwald prediction (gamma -> 0)
ratio_ana = ratio_vals[ratio_vals > 0]
sigma_xx_ana = (1 - ratio_ana) / (2 * (1 + ratio_ana))
sigma_pp_ana = (1 + ratio_ana) / (2 * (1 - ratio_ana))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(ratio_vals, sigma_xx_vals, 'b-', lw=2, label='Lyapunov (numerical)')
ax1.plot(ratio_ana, sigma_xx_ana, 'k--', lw=1.5, label='Kronwald formula')
ax1.axhline(0.5, color='gray', ls=':', lw=1, label='Vacuum noise (½)')
ax1.set_xlabel(r'Coupling ratio $\nu/g$')
ax1.set_ylabel(r'$\sigma_{xx} = \langle x^2 \rangle$')
ax1.set_title('x quadrature (squeezed)')
ax1.legend(fontsize=9)
ax1.set_ylim(0, 0.55)
ax1.fill_between(ratio_vals, 0, 0.5, alpha=0.08, color='blue', label='sub-vacuum')

ax2.semilogy(ratio_vals, sigma_xx_vals, 'b-', lw=2, label=r'$\sigma_{xx}$ (numerical)')
ax2.semilogy(ratio_vals, sigma_pp_vals, 'r-', lw=2, label=r'$\sigma_{pp}$ (numerical)')
ax2.semilogy(ratio_ana, sigma_xx_ana, 'b--', lw=1.2, label='Kronwald formula')
ax2.semilogy(ratio_ana, sigma_pp_ana, 'r--', lw=1.2)
ax2.axhline(0.5, color='gray', ls=':', lw=1)
ax2.set_xlabel(r'Coupling ratio $\nu/g$')
ax2.set_ylabel(r'$\sigma$ (log scale)')
ax2.set_title('Both quadratures (log scale)')
ax2.legend(fontsize=9)

plt.suptitle(r'Kronwald squeezing: $g=1$, $\kappa=1$, $\gamma=0.001$, $T=0$', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Target a specific squeezing: run the optimizer

The optimizer finds $(g, \nu)$ that minimise $\|\sigma_{\rm mech} - \sigma_{\rm target}\|^2$.

In [ ]:
target_r = 1.0   # change this to try different squeezing levels

g_opt, nu_opt, info = optimize(
    target_r, kappa=1.0, gamma=0.001, n_th=0.0,
    num_restarts=20, seed=42,
)

print(f'Target squeezing:  r = {target_r}')
print(f'Achieved squeezing: r = {info["r_achieved"]:.4f}')
print(f'Kronwald prediction: r = {info["r_Kronwald_analytic"]:.4f}  (γ→0 limit)')
print(f'\nOptimal parameters:')
print(f'  g  = {g_opt:.4f} κ')
print(f'  ν  = {nu_opt:.4f} κ')
print(f'  ν/g = {info["nu_over_g"]:.4f}  (theory: tanh(r) = {info["tanh_r_target"]:.4f})')
print(f'\nResidual loss: {info["loss"]:.2e}')

In [ ]:
# Plot achieved vs target covariance ellipses
sigma_ach = info['sigma_mech_achieved']
sigma_tgt = info['sigma_mech_target']

theta = np.linspace(0, 2 * np.pi, 300)
circle = np.array([np.cos(theta), np.sin(theta)])

def ellipse_from_cov(sigma):
    """Points on the 1-sigma covariance ellipse."""
    vals, vecs = np.linalg.eigh(sigma)
    return vecs @ (np.sqrt(vals)[:, None] * circle)

fig, ax = plt.subplots(figsize=(6, 5))

# Vacuum ellipse (circle)
r_vac = np.sqrt(0.5)
ax.plot(r_vac * np.cos(theta), r_vac * np.sin(theta),
        'gray', ls='--', lw=1.5, label='Vacuum noise (½ I)')

# Target
e_tgt = ellipse_from_cov(sigma_tgt)
ax.plot(e_tgt[0], e_tgt[1], 'k-', lw=2, label=f'Target  r={target_r}')

# Achieved
e_ach = ellipse_from_cov(sigma_ach)
ax.fill(e_ach[0], e_ach[1], alpha=0.25, color='steelblue')
ax.plot(e_ach[0], e_ach[1], 'steelblue', lw=2,
        label=f'Achieved r={info["r_achieved"]:.3f}')

ax.set_aspect('equal')
ax.set_xlabel(r'$x_{\rm mech}$')
ax.set_ylabel(r'$p_{\rm mech}$')
ax.set_title(f'Phase-space covariance ellipses   (target r={target_r})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Squeezing vs target r — optimizer vs Kronwald formula

Run the optimizer for several target values of $r$ and compare to the analytical prediction.

In [ ]:
r_targets = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
r_achieved_list = []
r_kron_list     = []
nu_over_g_list  = []

for r_t in r_targets:
    _, _, inf = optimize(r_t, gamma=0.001, num_restarts=15, seed=0)
    r_achieved_list.append(inf['r_achieved'])
    r_kron_list.append(inf['r_Kronwald_analytic'])
    nu_over_g_list.append(inf['nu_over_g'])
    print(f'  r_target={r_t:.2f}  r_achieved={inf["r_achieved"]:.4f}  '
          f'ν/g={inf["nu_over_g"]:.4f}  (tanh={np.tanh(r_t):.4f})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(r_targets, r_targets, 'k--', lw=1, label='Perfect match')
ax1.plot(r_targets, r_achieved_list, 'bo-', lw=1.5, ms=6, label='Optimizer result')
ax1.plot(r_targets, r_kron_list, 'r^--', lw=1.5, ms=5, label='Kronwald formula (γ→0)')
ax1.set_xlabel('Target squeezing $r$')
ax1.set_ylabel('Achieved squeezing $r$')
ax1.set_title('Optimizer vs Kronwald prediction')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(r_targets, np.tanh(r_targets), 'k--', lw=1.5, label=r'$\tanh(r)$  [ideal]')
ax2.plot(r_targets, nu_over_g_list, 'bo-', lw=1.5, ms=6, label=r'Optimised $\nu/g$')
ax2.set_xlabel('Target squeezing $r$')
ax2.set_ylabel(r'Coupling ratio $\nu/g$')
ax2.set_title(r'Optimal $\nu/g$ follows $\tanh(r)$')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle(r'Kronwald squeezing: $\gamma = 0.001\,\kappa$, $T=0$', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Effect of finite temperature and mechanical loss

For finite $\gamma$ and thermal phonon occupancy $\bar{n}_{\rm th}$, the squeezing degrades.

In [ ]:
r_target  = 1.0
g_val, nu_val = kronwald_params_from_r(r_target, g=1.0)

# Sweep gamma / kappa
gamma_vals = np.logspace(-4, -0.5, 40)
r_vs_gamma = []
for gm in gamma_vals:
    sm = np.array(mechanical_cov(jnp.array(
        solve_lyapunov(g_val, nu_val, kappa=1.0, gamma=gm, n_th=0.0))))
    r_vs_gamma.append(squeezing_from_cov(sm))

# Sweep n_th
n_th_vals = np.logspace(-2, 2, 40)
r_vs_nth = []
for n in n_th_vals:
    sm = np.array(mechanical_cov(jnp.array(
        solve_lyapunov(g_val, nu_val, kappa=1.0, gamma=0.001, n_th=n))))
    r_vs_nth.append(squeezing_from_cov(sm))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.semilogx(gamma_vals, r_vs_gamma, 'b-', lw=2)
ax1.axhline(r_target, color='k', ls='--', lw=1, label=f'Ideal r={r_target}')
ax1.set_xlabel(r'Mechanical decay $\gamma / \kappa$')
ax1.set_ylabel('Achieved squeezing $r$')
ax1.set_title(r'Squeezing degrades with $\gamma$  ($n_{\rm th}=0$)')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.semilogx(n_th_vals, r_vs_nth, 'r-', lw=2)
ax2.axhline(r_target, color='k', ls='--', lw=1, label=f'Ideal r={r_target}')
ax2.set_xlabel(r'Mean thermal phonons $\bar{n}_{\rm th}$')
ax2.set_ylabel('Achieved squeezing $r$')
ax2.set_title(r'Squeezing degrades with temperature  ($\gamma=0.001$)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle(rf'Robustness: $g={g_val}$, $\nu={nu_val:.4f}$, target $r={r_target}$', fontsize=12)
plt.tight_layout()
plt.show()